# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import json
import os
from pathlib import Path
from datetime import datetime
from typing import Dict, List

import chromadb
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import AIMessage, ToolMessage, UserMessage
from lib.tooling import tool
from lib.vector_db import VectorStore, VectorStoreManager

In [3]:
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [4]:
# Reuse persistent collection from Part 1
db = VectorStoreManager(os.getenv("OPENAI_API_KEY"))
db.chroma_client = chromadb.PersistentClient(path="chromadb")

collection = db.chroma_client.get_collection(
    name="udaplay",
    embedding_function=db.embedding_function
)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> List[Dict]:
    """
    Semantic search: Finds most relevant results in the vector DB.
    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["metadatas", "documents", "distances"]
    )
    
    # Extract relevant information from the results
    games_info = []
    for i in range(len(results['documents'][0])):
        metadata = results['metadatas'][0][i]
        game_data = {
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name", results['documents'][0][i]),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Publisher": metadata.get("Publisher"),
            "Genre": metadata.get("Genre"),
            "Description": metadata.get("Description")
        }
        games_info.append(game_data)
    
    return games_info

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Description about the evaluation result")

@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> Dict:
    """
    Based on the user's question and on the list of retrieved documents,
    analyzes the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge = LLM(model="gpt-4o-mini", temperature=0.0)
    prompt = (
        "Your task is to evaluate if the retrieved game documents contain enough factual evidence "
        "to respond to the query. Give a detailed explanation, so it's possible to take an action "
        "to accept it or not. Mark useful=False if the documents are irrelevant or incomplete.\n\n"
        f"Question: {question}\nRetrieved documents: {json.dumps(retrieved_docs)}"
    )

    evaluation = judge.invoke(prompt, response_format=EvaluationReport)
    return EvaluationReport.model_validate_json(evaluation.content).model_dump()

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> Dict:
    """
    Web search: Searches the web using Tavily for up-to-date or missing game industry information.
    args:
    - question: a question about the game industry.
    """
    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    search_results = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True
    )
    return search_results

### Agent

In [8]:
tools = [retrieve_game, evaluate_retrieval, game_web_search]

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions=(
        "You are UdaPlay, an AI Research Agent for the video game industry.\n"
        "You follow a pattern of Thought and Action, always explaining your reasoning before calling a tool.\n"
        "Follow this exact workflow to answer user questions:\n"
        "1. First, state your `Thought:` and call `retrieve_game` to search the internal vector database for relevant game information.\n"
        "2. Next, state your `Thought:` and call `evaluate_retrieval` with the original user question and a summary/JSON of the retrieved documents to assess whether they are sufficient.\n"
        "3. If `evaluate_retrieval` returns `useful: false` (or the information is incomplete), state your `Thought:` and call `game_web_search` to find accurate information on the web.\n"
        "4. Finally, provide a clear, structured final answer that includes:\n"
        "   - **Reasoning**: A brief summary of how the answer was found (internal knowledge base vs. web search fallback).\n"
        "   - **Answer**: The comprehensive answer to the user's question.\n"
        "   - **Citation / Source**: Cite the exact game document from the internal VectorDB (e.g., `[Source: Internal VectorDB - <Game Name> (<Platform>, <Year>)]`) or the web search URLs (e.g., `[Source: <Title> (<URL>)]`)."
    ),
    temperature=0.3,
)

In [9]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
from lib.state_machine import Run

def report_run(run_object: Run, query: str) -> Dict:
    """
    Formats and prints a structured report of an Agent run, including:
    - Query & Session metadata
    - Step-by-step Agent Reasoning & Tool Usage
    - Final Answer with Citations
    """
    final_state = run_object.get_final_state()
    messages = final_state.get("messages", [])
    session_id = final_state.get("session_id", "default")
    total_tokens = final_state.get("total_tokens", 0)

    # Isolate messages from the latest user query turn
    turn_start_idx = 0
    for idx in range(len(messages) - 1, -1, -1):
        if isinstance(messages[idx], UserMessage) and messages[idx].content == query:
            turn_start_idx = idx
            break
    turn_messages = messages[turn_start_idx:]

    reasoning_steps = []
    tool_usages = []
    final_answer = ""

    print("=" * 80)
    print(f"UDAPLAY AGENT REPORT")
    print("=" * 80)
    print(f"Query        : {query}")
    print(f"Session ID   : {session_id} | Run ID: {run_object.run_id} | Tokens: {total_tokens}")
    print("-" * 80)
    print("1. REASONING & TOOL USAGE TRACE:")

    step_num = 1
    for msg in turn_messages:
        if isinstance(msg, AIMessage):
            if msg.tool_calls:
                if msg.content:
                    reasoning_steps.append(msg.content)
                    print(f"\n  [Step {step_num} - Agent Reasoning]")
                    print(f"  {msg.content}")
                for tc in msg.tool_calls:
                    fn_name = tc.function.name
                    fn_args = tc.function.arguments
                    tool_usages.append({"tool": fn_name, "arguments": fn_args})
                    print(f"\n  [Step {step_num} - Tool Call] -> `{fn_name}`")
                    print(f"  Arguments: {fn_args}")
                step_num += 1
            else:
                final_answer = msg.content or ""
        elif isinstance(msg, ToolMessage):
            preview = msg.content
            if len(preview) > 350:
                preview = preview[:350] + "... [truncated]"
            print(f"  [Tool Output (`{msg.name}`)]: {preview}")

    print("\n" + "-" * 80)
    print("2. FINAL ANSWER & CITATIONS:")
    print("-" * 80)
    print(final_answer)
    print("=" * 80 + "\n")

    return {
        "query": query,
        "session_id": session_id,
        "reasoning": reasoning_steps,
        "tools_used": tool_usages,
        "final_answer": final_answer,
        "total_tokens": total_tokens,
    }


# Run the three required example queries
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

reports = []
for i, q in enumerate(queries, start=1):
    run_obj = udaplay_agent.invoke(query=q, session_id=f"query_{i}")
    reports.append(report_run(run_obj, q))

follow_up_query = "What platform was that Mario game released on, and who published it?"
follow_up_run = udaplay_agent.invoke(query=follow_up_query, session_id="query_2")
reports.append(report_run(follow_up_run, follow_up_query))

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
UDAPLAY AGENT REPORT
Query        : When Pokémon Gold and Silver was released?
Session ID   : query_1 | Run ID: 97c1bc73-674e-49af-85fa-02d8386bb7dc | Tokens: 2788
--------------------------------------------------------------------------------
1. REASONING & TOOL USAGE TRACE:

  [Step 1 - Tool Call] -> `retrieve_game`
  Arguments: {"query":"Pokémon Gold and Silver release date"}
  [Tool Output (`retrieve_game`)]: "[{'Platform': 'Game Boy Color', 'Name': 'Pok\u00e9mon Gold and Silver', 'YearOfRelease': 1999, 'Publisher': 'Nintendo', 'Genre': 'Role-playing', 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9m